In [2]:
# ============================================================================
# SCRIPT: Daily-Averaged PSD Stacking by Mass, Eddington, and T_max Diagonals
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patheffects
import warnings
from astropy.timeseries import LombScargle

warnings.filterwarnings('ignore')

# ---------------------- USER CONFIG (Edit here) ----------------------

base_dir = r"C:\Users\vitma\Documents\GitHub\AGN-Power-Spectra\ztffp_final_data_2025" 
csv_meta_path = r"Cross_15_luiz.csv"

# --- BANDS TO PROCESS ---
BANDS_TO_PROCESS = ['g', 'r']

# Base Output Directories
plots_dir_base = "plots_Stacking"

# --- UNIT AUTOMATION ---
FREQ_UNIT = 'days'
MIN_CONTRIBS_PER_FREQ = 1              # Minimum number of contributions per frequency bin

# --- AUTOMATED BIN WIDTH (PHYSICS-BASED) ---
MIN_CADENCE_DAYS = 1.0 # Trava de segurança física (cadência diária forçada)
TARGET_BINS = 500 

time_multiplier = 1.0
freq_label = r"day$^{-1}$"
f_max_nyquist = 1.0 / (2.0 * MIN_CADENCE_DAYS) # Máximo será 0.5 dia^-1
BIN_WIDTH = f_max_nyquist / TARGET_BINS  

# Normalization
NORMALIZATION_MODE = 'psd'

# Master Grid bins
mass_edges = [7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]
edd_edges  = [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0]

dpi_save = 200
cmap = matplotlib.colormaps.get_cmap('tab10')

# =============================================================================
# 0. METADATA CROSSMATCH
# =============================================================================
print("Lendo catálogo de metadados...")
csv_meta_path = csv_meta_path.replace("\\", "/")
meta = pd.read_csv(csv_meta_path)

ra_cat = meta['RA'].values
dec_cat = meta['DEC'].values
logmbh_cat = meta['LOGMBH'].values
logedd_cat = meta['LOGLEDD_RATIO'].values

# =============================================================================
# MAIN LOOP OVER BANDS
# =============================================================================
for band in BANDS_TO_PROCESS:
    print(f"\n{'='*60}")
    print(f"PROCESSING BAND: '{band}'")
    print(f"{'='*60}")
    
    band_plots_dir = os.path.join(plots_dir_base, f"band_{band}")
    os.makedirs(band_plots_dir, exist_ok=True)

    # =============================================================================
    # 1. FILTERING & DAILY AVERAGING
    # =============================================================================
    folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and "plots" not in f]
    valid_data = []

    for folder in tqdm(folders, desc=f"Filtrando e tirando média diária (Band {band})"):
        try:
            ra_f, dec_f = map(float, folder.split('_'))
        except: continue
        
        # Euclidean Crossmatch
        dist = np.sqrt((ra_cat - ra_f)**2 + (dec_cat - dec_f)**2)
        idx = np.argmin(dist)
        if dist[idx] > 1e-4: continue
        
        m, e = logmbh_cat[idx], logedd_cat[idx]
        if np.isnan(m) or np.isnan(e): continue

        # Outlier filter
        if m < mass_edges[0] or m >= mass_edges[-1]: continue
        if e < edd_edges[0] or e >= edd_edges[-1]: continue

        txt_files = glob.glob(os.path.join(base_dir, folder, f"*_ztf_{band}_filtered.txt"))
        if not txt_files: continue
        
        try:
            df = pd.read_csv(txt_files[0])
            if len(df) < 10 or 'mjd' not in df.columns or 'flux' not in df.columns: continue
            
            # --- CORTE: MÉDIA INTRA-DIA ---
            # Isola a parte inteira do dia para agrupar
            df['mjd_int'] = np.floor(df['mjd']).astype(int)
            # Tira a média das observações daquele mesmo dia
            df_daily = df.groupby('mjd_int').agg({'mjd': 'mean', 'flux': 'mean'}).reset_index(drop=True)
            
            if len(df_daily) < 10: continue
            
            t = df_daily['mjd'].values
            flux = df_daily['flux'].values
        except: continue
        
        T = t.max() - t.min()
        dt = np.min(np.diff(t))
        
        valid_data.append({
            'label': folder, 't': t, 'flux': flux, 'T': T, 'dt': dt,
            'LOGMBH': m, 'LOGLEDD_RATIO': e
        })

    if not valid_data:
        print(f"AVISO: Nenhum dado válido encontrado para a banda '{band}'. Pulando para a próxima...")
        continue

    # Master Grid Limits for this specific band
    T_max = max([d['T'] for d in valid_data])
    global_fmin = 1.0 / (T_max * time_multiplier)

    N_bins = int(np.ceil((f_max_nyquist - global_fmin) / BIN_WIDTH))
    bin_edges = global_fmin + np.arange(N_bins + 1) * BIN_WIDTH
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    binned_matrix = np.full((len(valid_data), N_bins), np.nan)

    # =============================================================================
    # 2. AUTOPOWER PSD CALCULATION & BINNING
    # =============================================================================
    for i, d in enumerate(tqdm(valid_data, desc=f"Computing Autopower & Binning (Band {band})")):
        t_scaled = (d['t'] - d['t'].min()) * time_multiplier
        flux = d['flux']
        
        # --- NORMALIZAÇÃO DA CURVA DE LUZ ANTES DA PSD ---
        y_norm = (flux - np.median(flux)) / np.median(flux)
        
        # O LombScargle recebe a curva já normalizada e é travado na frequência de Nyquist (0.5 dia^-1)
        ls = LombScargle(t_scaled, y_norm)
        freq_auto, power_auto = ls.autopower(normalization=NORMALIZATION_MODE, maximum_frequency=f_max_nyquist)
        
        inds = np.searchsorted(bin_edges, freq_auto, side='right') - 1
        valid_mask = (inds >= 0) & (inds < N_bins)
        
        if np.any(valid_mask):
            bid = inds[valid_mask]
            pw = power_auto[valid_mask]
            
            sums = np.bincount(bid, weights=pw, minlength=N_bins)
            counts = np.bincount(bid, minlength=N_bins)
            
            means = np.full(N_bins, np.nan)
            mask = counts > 0
            means[mask] = sums[mask] / counts[mask]
            binned_matrix[i, :] = means

    # =============================================================================
    # 3. MARGINALIZED STACKING (MASS, EDDINGTON & T_MAX DIAGONALS)
    # =============================================================================
    print(f"Gerando gráficos de Stacking (Band {band})...")
    
    LOGMBH_array = np.array([d['LOGMBH'] for d in valid_data])
    LOGLEDD_array = np.array([d['LOGLEDD_RATIO'] for d in valid_data])

    # --- 3.1 Stack by Mass ---
    plt.figure(figsize=(9,6))
    plotted_mass = 0
    for mi in range(len(mass_edges)-1):
        mask_m = (LOGMBH_array >= mass_edges[mi]) & (LOGMBH_array < mass_edges[mi+1])
        idxs = np.where(mask_m)[0]
        
        if len(idxs) < 1: continue
        
        arr = binned_matrix[idxs, :]
        good_mask = np.sum(~np.isnan(arr), axis=0) >= MIN_CONTRIBS_PER_FREQ
        
        if np.sum(good_mask) < 2: continue
        
        med = np.full(N_bins, np.nan)
        med[good_mask] = np.nanmedian(arr[:, good_mask], axis=0)
        
        x = np.log10(bin_centers[good_mask])
        y = np.log10(med[good_mask])
        label = f"{mass_edges[mi]:.1f} <= log M < {mass_edges[mi+1]:.1f} (n={len(idxs)})"
        plt.plot(x, y, label=label, color=cmap(mi % cmap.N), linewidth=1.8, alpha=0.9)
        plotted_mass += 1
                 
    if plotted_mass > 0:
        plt.xlabel(f"log10(frequency [{freq_label}])", fontweight='bold')
        plt.ylabel("log10(Power)", fontweight='bold')
        plt.title(f"Stacked PSD by Mass Intervals (Band {band})", fontweight='bold')
        plt.legend(loc='best', fontsize='small')
        plt.grid(True, ls=':', alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(band_plots_dir, f"stack_by_mass_band_{band}.png"), dpi=dpi_save)
    plt.close()

    # --- 3.2 Stack by Eddington ---
    plt.figure(figsize=(9,6))
    plotted_edd = 0
    for ej in range(len(edd_edges)-1):
        mask_e = (LOGLEDD_array >= edd_edges[ej]) & (LOGLEDD_array < edd_edges[ej+1])
        idxs = np.where(mask_e)[0]
        
        if len(idxs) < 1: continue
        
        arr = binned_matrix[idxs, :]
        good_mask = np.sum(~np.isnan(arr), axis=0) >= MIN_CONTRIBS_PER_FREQ
        
        if np.sum(good_mask) < 2: continue
        
        med = np.full(N_bins, np.nan)
        med[good_mask] = np.nanmedian(arr[:, good_mask], axis=0)
        
        x = np.log10(bin_centers[good_mask])
        y = np.log10(med[good_mask])
        label = f"{edd_edges[ej]:.1f} <= log R_Edd < {edd_edges[ej+1]:.1f} (n={len(idxs)})"
        plt.plot(x, y, label=label, color=cmap(ej % cmap.N), linewidth=1.8, alpha=0.9)
        plotted_edd += 1
                 
    if plotted_edd > 0:
        plt.xlabel(f"log10(frequency [{freq_label}])", fontweight='bold')
        plt.ylabel("log10(Power)", fontweight='bold')
        plt.title(f"Stacked PSD by Eddington Intervals (Band {band})", fontweight='bold')
        plt.legend(loc='best', fontsize='small')
        plt.grid(True, ls=':', alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(band_plots_dir, f"stack_by_edd_band_{band}.png"), dpi=dpi_save)
    plt.close()

    # --- 3.3 Stack by T_max (Diagonal Tracks) ---
    plt.figure(figsize=(9,6))
    plotted_tmax = 0
    
    # Determina o índice do bin de cada objeto
    mi_array = np.digitize(LOGMBH_array, mass_edges) - 1
    ej_array = np.digitize(LOGLEDD_array, edd_edges) - 1
    
    # A diferença (ej - mi) agrupa perfeitamente as diagonais da matriz
    T_track_array = ej_array - mi_array
    
    # Máscara para garantir que os objetos estão dentro do grid
    valid_grid = (mi_array >= 0) & (mi_array < len(mass_edges)-1) & \
                 (ej_array >= 0) & (ej_array < len(edd_edges)-1)
                 
    unique_tracks = np.unique(T_track_array[valid_grid])
    
    for track in sorted(unique_tracks):
        idxs = np.where(valid_grid & (T_track_array == track))[0]
        
        if len(idxs) < 1: continue
        
        arr = binned_matrix[idxs, :]
        good_mask = np.sum(~np.isnan(arr), axis=0) >= MIN_CONTRIBS_PER_FREQ
        
        if np.sum(good_mask) < 2: continue
        
        med = np.full(N_bins, np.nan)
        med[good_mask] = np.nanmedian(arr[:, good_mask], axis=0)
        
        x = np.log10(bin_centers[good_mask])
        y = np.log10(med[good_mask])
        
        # Cálculo nominal do proxy de T_max para a legenda: log(L/L_edd) - log(M)
        # O bin central E_c - M_c escala com -9.5 + track * 0.5
        ratio_val = -9.5 + track * 0.5
        
        label = f"log($\lambda_{{Edd}}$/M) $\\approx$ {ratio_val:.1f} (n={len(idxs)})"
        
        color_idx = (track + 10) % cmap.N
        plt.plot(x, y, label=label, color=cmap(color_idx), linewidth=1.8, alpha=0.9)
        plotted_tmax += 1
                 
    if plotted_tmax > 0:
        plt.xlabel(f"log10(frequency [{freq_label}])", fontweight='bold')
        plt.ylabel("log10(Power)", fontweight='bold')
        plt.title(f"Stacked PSD by Disc Temperature ($T_{{max}}$) Tracks (Band {band})", fontweight='bold')
        plt.legend(loc='best', fontsize='small')
        plt.grid(True, ls=':', alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(band_plots_dir, f"stack_by_tmax_band_{band}.png"), dpi=dpi_save)
    plt.close()


    # =============================================================================
    # 4. FINAL DYNAMIC GRID PLOT (Mass vs Eddington Distribution)
    # =============================================================================
    print(f"Gerando grid de distribuição (Band {band})...")
    
    xe = np.array(mass_edges)
    ye = np.array(edd_edges)
    
    # Calcular o histograma 2D para contar os objetos em cada bin
    H, _, _ = np.histogram2d(LOGMBH_array, LOGLEDD_array, bins=(xe, ye))
    H = H.T  # Transpõe para o eixo X ser Massa e o Y ser Eddington

    fig, ax = plt.subplots(figsize=(11, 9))
    
    # Plota a malha térmica de fundo (heatmap) para densidade visual
    ax.pcolormesh(xe, ye, H, cmap='Blues', alpha=0.5, edgecolor='black', linewidth=1.2, zorder=1)
    
    # Plota os objetos individuais na grade
    ax.scatter(LOGMBH_array, LOGLEDD_array, c='gray', s=15, alpha=0.7, edgecolors='k', linewidths=0.3, zorder=2)

    # Escreve o número exato de objetos dentro de cada bloco
    for i in range(len(ye)-1):
        for j in range(len(xe)-1):
            count = int(H[i,j])
            if count > 0:
                txt = ax.text((xe[j]+xe[j+1])/2, (ye[i]+ye[i+1])/2, str(count), 
                              ha='center', va='center', fontweight='bold', fontsize=14, zorder=10)
                txt.set_path_effects([matplotlib.patheffects.withStroke(linewidth=3, foreground='white')])

    ax.set_xticks(xe)
    ax.set_yticks(ye)
    ax.set_xlabel(r'$\log M_{\rm{BH}} \ [M_{\odot}]$', fontsize=16, fontweight='bold')
    ax.set_ylabel(r'$\log (L_{\rm{bol}}/L_{\rm{Edd}})$', fontsize=16, fontweight='bold')
    ax.set_title(f"Quasar Distribution in Mass-Eddington Grid (Band {band})", fontweight='bold', fontsize=16)
    
    for tl in ax.get_xticklabels() + ax.get_yticklabels(): 
        tl.set_fontweight('bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(band_plots_dir, f"grid_counts_band_{band}.png"), dpi=dpi_save)
    plt.close()

print("\nProcessamento completo para todas as bandas!")

Lendo catálogo de metadados...

PROCESSING BAND: 'g'


Computing Autopower & Binning (Band g): 100%|██████████| 464/464 [00:01<00:00, 257.61it/s]


Gerando gráficos de Stacking (Band g)...
Gerando grid de distribuição (Band g)...

PROCESSING BAND: 'r'


Computing Autopower & Binning (Band r): 100%|██████████| 461/461 [00:01<00:00, 369.82it/s]


Gerando gráficos de Stacking (Band r)...
Gerando grid de distribuição (Band r)...

Processamento completo para todas as bandas!
